# 02 · Feature Engineering

Purpose:

1. Load raw OHLCV data independently from Notebook 01.
2. Build historical-only predictive features.
3. Add same-day benchmark context.
4. Create a five-trading-day future target.
5. Validate and persist the ML-ready dataset.

**Prediction timing assumption:** predictions are made after the market close on day `t`. Therefore day `t` price, volume, and benchmark information are allowed as features. No feature may use data from `t+1` onward.

In [1]:
from google.colab import drive
from pathlib import Path

import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/ai-tech-market-risk")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [
    RAW_DATA_DIR,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    MODEL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

RAW_DATA_FILE = RAW_DATA_DIR / "market_data.csv"
PROCESSED_DATA_FILE = PROCESSED_DATA_DIR / "ml_features.csv"

print("Project root:", PROJECT_ROOT)
print("Raw input:", RAW_DATA_FILE)
print("Processed output:", PROCESSED_DATA_FILE)

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
STOCKS = [
    "NVDA",
    "AMD",
    "MSFT",
    "GOOGL",
    "META",
]

BENCHMARKS = [
    "SPY",
    "QQQ",
    "SMH",
]

TICKERS = STOCKS + BENCHMARKS

RAW_COLUMNS = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Ticker",
]

LOOKBACK_PERIODS = [5, 10, 20]
VOLATILITY_WINDOWS = [5, 10, 20]
MOVING_AVERAGE_WINDOWS = [5, 20]

TARGET_HORIZON_DAYS = 5
LARGE_MOVE_THRESHOLD = 0.03

## Load and validate raw data

Notebook 02 deliberately recalculates `Return_1D`. It does not depend on any feature created in Notebook 01.

In [ ]:
if not RAW_DATA_FILE.exists():
    raise FileNotFoundError(
        f"Raw dataset not found: {RAW_DATA_FILE}. Run Notebook 01 first."
    )

market_data = pd.read_csv(
    RAW_DATA_FILE,
    parse_dates=["Date"],
)

missing_columns = set(RAW_COLUMNS) - set(market_data.columns)
if missing_columns:
    raise ValueError(
        f"Raw dataset is missing columns: {sorted(missing_columns)}"
    )

# Keep only the true raw schema.
# This also makes the notebook robust to an older raw file that accidentally
# contains derived columns such as Return_1D.
market_data = market_data[RAW_COLUMNS].copy()

market_data = (
    market_data
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
)

duplicates = market_data.duplicated(["Ticker", "Date"]).sum()
missing_raw = market_data[RAW_COLUMNS].isna().sum().sum()
observed_tickers = set(market_data["Ticker"].unique())

print("Shape:", market_data.shape)
print("Duplicate ticker/date rows:", duplicates)
print("Missing raw values:", missing_raw)
print("Tickers:", sorted(observed_tickers))

if duplicates != 0:
    raise ValueError("Duplicate ticker/date rows detected.")

if missing_raw != 0:
    raise ValueError("Missing values detected in raw data.")

if observed_tickers != set(TICKERS):
    raise ValueError("Raw dataset does not contain the expected ticker universe.")

market_data.head()

## Historical return features

All return features look backward from day `t`, so they are safe predictors.

In [ ]:
market_features = market_data.copy()

market_features["Return_1D"] = (
    market_features
    .groupby("Ticker")["Close"]
    .transform(
        lambda x: x.pct_change(fill_method=None)
    )
)

for period in LOOKBACK_PERIODS:
    market_features[f"Return_{period}D"] = (
        market_features
        .groupby("Ticker")["Close"]
        .transform(
            lambda x, p=period: x.pct_change(
                periods=p,
                fill_method=None,
            )
        )
    )

## Rolling volatility

Rolling standard deviation measures how variable recent daily returns have been.

In [ ]:
for window in VOLATILITY_WINDOWS:
    market_features[f"Volatility_{window}D"] = (
        market_features
        .groupby("Ticker")["Return_1D"]
        .transform(
            lambda x, w=window: x.rolling(
                window=w,
                min_periods=w,
            ).std()
        )
    )

## Volume features

Relative volume is easier to compare across stocks than raw share volume.

In [ ]:
market_features["Volume_Change_1D"] = (
    market_features
    .groupby("Ticker")["Volume"]
    .transform(
        lambda x: x.pct_change(fill_method=None)
    )
)

market_features["Volume_MA_20D"] = (
    market_features
    .groupby("Ticker")["Volume"]
    .transform(
        lambda x: x.rolling(
            window=20,
            min_periods=20,
        ).mean()
    )
)

market_features["Relative_Volume_20D"] = (
    market_features["Volume"]
    / market_features["Volume_MA_20D"]
)

## Price position relative to moving averages

Using a ratio rather than the raw moving-average value makes the feature comparable across stocks with different price levels.

In [ ]:
for window in MOVING_AVERAGE_WINDOWS:
    moving_average = (
        market_features
        .groupby("Ticker")["Close"]
        .transform(
            lambda x, w=window: x.rolling(
                window=w,
                min_periods=w,
            ).mean()
        )
    )

    market_features[f"Price_vs_MA_{window}D"] = (
        market_features["Close"] / moving_average - 1
    )

## Benchmark context

SPY represents the broad US market, QQQ gives technology/growth-heavy market context, and SMH provides semiconductor-sector context.

In [ ]:
benchmark_data = (
    market_features[
        market_features["Ticker"].isin(BENCHMARKS)
    ][
        [
            "Date",
            "Ticker",
            "Return_1D",
            "Return_5D",
        ]
    ]
    .copy()
)

benchmark_1d = (
    benchmark_data
    .pivot(
        index="Date",
        columns="Ticker",
        values="Return_1D",
    )
    .rename(
        columns={
            "SPY": "SPY_Return_1D",
            "QQQ": "QQQ_Return_1D",
            "SMH": "SMH_Return_1D",
        }
    )
)

benchmark_5d = (
    benchmark_data
    .pivot(
        index="Date",
        columns="Ticker",
        values="Return_5D",
    )
    .rename(
        columns={
            "SPY": "SPY_Return_5D",
            "QQQ": "QQQ_Return_5D",
            "SMH": "SMH_Return_5D",
        }
    )
)

benchmark_features = (
    benchmark_1d
    .join(benchmark_5d)
    .reset_index()
)

benchmark_features.columns.name = None

benchmark_features.head()

In [ ]:
stock_data = (
    market_features[
        market_features["Ticker"].isin(STOCKS)
    ]
    .copy()
)

stock_data = stock_data.merge(
    benchmark_features,
    on="Date",
    how="left",
    validate="many_to_one",
)

stock_data = (
    stock_data
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
)

stock_data["Excess_vs_QQQ_1D"] = (
    stock_data["Return_1D"]
    - stock_data["QQQ_Return_1D"]
)

stock_data["Excess_vs_SMH_1D"] = (
    stock_data["Return_1D"]
    - stock_data["SMH_Return_1D"]
)

## Five-day future target

The target is allowed to look forward because it is the outcome we want the model to predict.

For the MVP, `Large_Move_5D = 1` means the absolute five-trading-day return exceeds 3%. This is a fixed large-move threshold, not yet a volatility-normalized definition of an abnormal move.

In [ ]:
future_close = (
    stock_data
    .groupby("Ticker")["Close"]
    .shift(-TARGET_HORIZON_DAYS)
)

stock_data["Future_Return_5D"] = (
    future_close / stock_data["Close"] - 1
)

stock_data["Large_Move_5D"] = np.where(
    stock_data["Future_Return_5D"].isna(),
    np.nan,
    (
        stock_data["Future_Return_5D"].abs()
        > LARGE_MOVE_THRESHOLD
    ).astype(int),
)

In [ ]:
stock_data[
    stock_data["Ticker"] == "NVDA"
][
    [
        "Date",
        "Close",
        "Future_Return_5D",
        "Large_Move_5D",
    ]
].tail(10)

## Build the ML-ready table

In [ ]:
FEATURE_COLUMNS = [
    "Return_1D",
    "Return_5D",
    "Return_10D",
    "Return_20D",
    "Volatility_5D",
    "Volatility_10D",
    "Volatility_20D",
    "Volume_Change_1D",
    "Relative_Volume_20D",
    "Price_vs_MA_5D",
    "Price_vs_MA_20D",
    "SPY_Return_1D",
    "QQQ_Return_1D",
    "SMH_Return_1D",
    "SPY_Return_5D",
    "QQQ_Return_5D",
    "SMH_Return_5D",
    "Excess_vs_QQQ_1D",
    "Excess_vs_SMH_1D",
]

ML_COLUMNS = [
    "Date",
    "Ticker",
    "Close",
    *FEATURE_COLUMNS,
    "Future_Return_5D",
    "Large_Move_5D",
]

ml_data = stock_data[ML_COLUMNS].copy()

numeric_before_drop = ml_data[
    FEATURE_COLUMNS + ["Future_Return_5D"]
].to_numpy()

infinite_values = np.isinf(numeric_before_drop).sum()

print("Infinite values before cleaning:", infinite_values)

if infinite_values != 0:
    raise ValueError(
        "Infinite feature/target values detected. Investigate before training."
    )

ml_data = ml_data.dropna(
    subset=FEATURE_COLUMNS + ["Large_Move_5D"]
).copy()

ml_data["Large_Move_5D"] = (
    ml_data["Large_Move_5D"].astype("int8")
)

ml_data = (
    ml_data
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

## Validate feature table and class balance

In [ ]:
missing_ml_values = ml_data[
    FEATURE_COLUMNS + ["Large_Move_5D"]
].isna().sum().sum()

duplicate_samples = ml_data.duplicated(
    ["Ticker", "Date"]
).sum()

print("Rows:", len(ml_data))
print("Features:", len(FEATURE_COLUMNS))
print("Missing feature/target values:", missing_ml_values)
print("Duplicate ticker/date rows:", duplicate_samples)
print(
    "Date range:",
    ml_data["Date"].min().date(),
    "to",
    ml_data["Date"].max().date(),
)

if missing_ml_values != 0:
    raise ValueError("Missing values remain in the ML dataset.")

if duplicate_samples != 0:
    raise ValueError("Duplicate ML samples detected.")

In [ ]:
target_distribution = (
    ml_data["Large_Move_5D"]
    .value_counts()
    .sort_index()
)

target_percentage = (
    ml_data["Large_Move_5D"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

print("Counts:")
print(target_distribution)

print("\nPercentages:")
print(target_percentage.round(2))

ticker_target_rate = (
    ml_data
    .groupby("Ticker")["Large_Move_5D"]
    .agg(
        observations="count",
        large_moves="sum",
        large_move_rate="mean",
    )
)

ticker_target_rate["large_move_rate"] = (
    ticker_target_rate["large_move_rate"] * 100
)

ticker_target_rate

## Manual target sanity check

This verifies that the stored target equals the manually calculated return from the current close to the close five trading rows later for the same ticker.

In [ ]:
nvda_ml = ml_data[
    ml_data["Ticker"] == "NVDA"
]

if len(nvda_ml) <= 500:
    raise ValueError("Not enough NVDA rows for the selected sanity-check example.")

example = nvda_ml.iloc[500]

ticker_history = (
    stock_data[
        stock_data["Ticker"] == example["Ticker"]
    ]
    .sort_values("Date")
    .reset_index(drop=True)
)

row_index = ticker_history.index[
    ticker_history["Date"] == example["Date"]
][0]

current_price = ticker_history.loc[row_index, "Close"]
future_price = ticker_history.loc[
    row_index + TARGET_HORIZON_DAYS,
    "Close",
]

manual_future_return = (
    future_price / current_price - 1
)

stored_future_return = example["Future_Return_5D"]

print("Date:", example["Date"])
print("Current price:", current_price)
print(
    f"Price {TARGET_HORIZON_DAYS} trading days later:",
    future_price,
)
print("Manual future return:", manual_future_return)
print("Stored future return:", stored_future_return)
print(
    "Matches:",
    np.isclose(
        manual_future_return,
        stored_future_return,
        rtol=1e-12,
        atol=1e-12,
    ),
)

if not np.isclose(
    manual_future_return,
    stored_future_return,
    rtol=1e-12,
    atol=1e-12,
):
    raise ValueError("Future-return target sanity check failed.")

## Persist and verify the processed dataset

The file is written to Google Drive, checked for existence, reloaded, and compared with the in-memory dataset shape and schema.

In [ ]:
ml_data.to_csv(
    PROCESSED_DATA_FILE,
    index=False,
)

print("Saved:", PROCESSED_DATA_FILE)
print("Exists:", PROCESSED_DATA_FILE.exists())
print("Size:", PROCESSED_DATA_FILE.stat().st_size, "bytes")

In [ ]:
reloaded_ml_data = pd.read_csv(
    PROCESSED_DATA_FILE,
    parse_dates=["Date"],
)

print("Original shape:", ml_data.shape)
print("Reloaded shape:", reloaded_ml_data.shape)
print(
    "Reload successful:",
    reloaded_ml_data.shape == ml_data.shape,
)

if reloaded_ml_data.shape != ml_data.shape:
    raise ValueError("Reloaded processed dataset has the wrong shape.")

if reloaded_ml_data.columns.tolist() != ml_data.columns.tolist():
    raise ValueError("Reloaded processed dataset has the wrong schema.")

print("Feature engineering pipeline completed successfully.")